# Model anatomy and hidden states

This notebook prepares the activation-steering experiment by making the model's mechanics visible. It does **not** construct or apply a steering vector yet.

We will:

1. load a small decoder-only model on the best available PyTorch device;
2. inspect tokenization and padding;
3. inspect the decoder architecture and tensor shapes;
4. capture every layer's hidden state;
5. construct mean-pooled and final-token sentence representations;
6. use a read-only forward hook to locate the future intervention point;
7. establish an unsteered formal-rewrite generation baseline.

## Why this model?

We use `Qwen/Qwen2.5-0.5B-Instruct`. It is an instruction-tuned causal language model with approximately 0.49B parameters, 24 decoder layers, and hidden size 896. It is small enough for local experimentation while retaining a chat template and a reasonable chance of following a formal-rewriting instruction.

A small model is useful here because we will retain hidden states from every layer. Those activations can consume more memory than ordinary generation. Model quality is not the only criterion: the first model must be understandable and easy to probe.

In [2]:
from pathlib import Path
import platform
import sys

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

pd.set_option("display.max_colwidth", 120)
torch.manual_seed(42)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Machine:", platform.machine())
print("Python executable:", sys.executable)

Python: 3.11.0
PyTorch: 2.12.1
Transformers: 5.14.1
Machine: arm64
Python executable: /usr/local/bin/python3


## 1. Select the compute device

On an Apple Silicon Mac, `mps` should normally be selected. The CPU fallback keeps the notebook runnable in environments where the Metal device is unavailable. Float16 reduces model and activation memory on MPS; CPU uses float32 for compatibility.

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

MODEL_DTYPE = torch.float16 if device.type in {"mps", "cuda"} else torch.float32
print("Selected device:", device)
print("Model dtype:", MODEL_DTYPE)
print("MPS built into PyTorch:", torch.backends.mps.is_built())
print("MPS currently available:", torch.backends.mps.is_available())

Selected device: mps
Model dtype: torch.float16
MPS built into PyTorch: True
MPS currently available: True


## 2. Load the tokenizer and model

The first execution downloads the model from Hugging Face and caches it locally. Later executions reuse the cache. We load explicitly rather than using a high-level pipeline so every tensor remains accessible.

In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
)
model = model.to(device)
model.eval()

print("Loaded:", MODEL_ID)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Padding token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("EOS token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
Model device: mps:0
Model dtype: torch.float16
Padding token: '<|endoftext|>' 151643
EOS token: '<|im_end|>' 151645


## 3. Inspect size and architecture

For this architecture, `model.model.layers` is the decoder-layer stack. A future steering hook will target the residual-stream tensor returned by one of these layers.

In [5]:
config = model.config
parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
parameter_bytes = sum(
    parameter.numel() * parameter.element_size() for parameter in model.parameters()
)

architecture = pd.Series(
    {
        "architecture": config.architectures[0],
        "parameters": f"{parameter_count:,}",
        "trainable_parameters": f"{trainable_count:,}",
        "parameter_memory_GiB": parameter_bytes / 1024**3,
        "layers": config.num_hidden_layers,
        "hidden_size": config.hidden_size,
        "attention_heads": config.num_attention_heads,
        "key_value_heads": config.num_key_value_heads,
        "vocabulary_size": config.vocab_size,
        "maximum_positions": config.max_position_embeddings,
    },
    name="value",
)
architecture.to_frame()

,value
architecture,Qwen2ForCausalLM
parameters,"494,032,768"
trainable_parameters,"494,032,768"
parameter_memory_GiB,0.920208
layers,24
hidden_size,896
attention_heads,14
key_value_heads,2
vocabulary_size,151936
maximum_positions,32768


In [6]:
decoder_layers = model.model.layers
print("Decoder container:", type(model.model).__name__)
print("Layer container:", type(decoder_layers).__name__)
print("Number of decoder layers:", len(decoder_layers))
print("One decoder layer:", type(decoder_layers[0]).__name__)
print("Final normalization:", type(model.model.norm).__name__)
print("Language-model head:", type(model.lm_head).__name__)

Decoder container: Qwen2Model
Layer container: ModuleList
Number of decoder layers: 24
One decoder layer: Qwen2DecoderLayer
Final normalization: Qwen2RMSNorm
Language-model head: Linear


## 4. Load a few aligned GYAFC examples

Only a tiny sample is needed for model anatomy. Full activation extraction will be a separate notebook after the representation choices are understood.

In [7]:
DATA_ROOT = Path("/Users/nandanprince/Downloads/GYAFC_Corpus")
DOMAIN_DIRECTORIES = {
    "entertainment_music": "Entertainment_Music",
    "family_relationships": "Family_Relationships",
}


def read_first_pairs(domain: str, directory: str, n: int = 3) -> pd.DataFrame:
    train_directory = DATA_ROOT / directory / "train"
    informal = (train_directory / "informal").read_text(encoding="utf-8").splitlines()[:n]
    formal = (train_directory / "formal").read_text(encoding="utf-8").splitlines()[:n]
    assert len(informal) == len(formal) == n
    return pd.DataFrame({"domain": domain, "informal": informal, "formal": formal})


example_pairs = pd.concat(
    [read_first_pairs(domain, directory) for domain, directory in DOMAIN_DIRECTORIES.items()],
    ignore_index=True,
)
example_pairs

,domain,informal,formal
0,entertainment_music,the movie The In-Laws not exactly a holiday movie but funny and good!,"The In-Laws movie isn't a holiday movie, but it's okay."
1,entertainment_music,that page did not give me viroses(i think),I don't think that page gave me viruses.
2,entertainment_music,"of corse i be wachin it evry day, my fav charachter is Inuasha","I watch it everyday, my favorite charachter is Inuasha."
3,family_relationships,"Sure, it's ok, but I always have let the guy ask me.",I prefer to let the guy ask me.
4,family_relationships,"Hmmm, I'm a guy suffering from verbal abuse from my wife.",I suffer through verbal abuse from my wife.
5,family_relationships,You will have more friends that you want... ;),You will have more friends than you want.


## 5. Understand tokenization

A model does not receive words or sentences directly. It receives integer token IDs. Token boundaries can differ between paired rewrites, so we should not subtract token positions one by one. Instead, the first activation experiment will reduce each sentence to one representation before computing paired differences.

In [8]:
example_text = example_pairs.loc[0, "informal"]
tokenized_example = tokenizer(
    example_text,
    add_special_tokens=True,
    return_offsets_mapping=True,
)

token_table = pd.DataFrame(
    {
        "position": range(len(tokenized_example["input_ids"])),
        "token_id": tokenized_example["input_ids"],
        "token": tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"]),
        "character_offsets": tokenized_example["offset_mapping"],
    }
)
print("Original text:", example_text)
token_table

Original text: the movie The In-Laws not exactly a holiday movie but funny and good!


,position,token_id,token,character_offsets
0,0,1782,the,"(0, 3)"
1,1,5700,Ġmovie,"(3, 9)"
2,2,576,ĠThe,"(9, 13)"
3,3,758,ĠIn,"(13, 16)"
4,4,8125,-L,"(16, 18)"
5,5,8635,aws,"(18, 21)"
6,6,537,Ġnot,"(21, 25)"
7,7,6896,Ġexactly,"(25, 33)"
8,8,264,Ġa,"(33, 35)"
9,9,13257,Ġholiday,"(35, 43)"


In [9]:
representation_texts = (
    example_pairs["informal"].tolist() + example_pairs["formal"].tolist()
)
encoded = tokenizer(
    representation_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

print("input_ids shape:", tuple(encoded["input_ids"].shape))
print("attention_mask shape:", tuple(encoded["attention_mask"].shape))
print("non-padding tokens per sentence:", encoded["attention_mask"].sum(dim=1).tolist())
print("First padded token sequence:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

input_ids shape: (12, 22)
attention_mask shape: (12, 22)
non-padding tokens per sentence: [16, 12, 22, 16, 15, 10, 16, 10, 15, 9, 9, 9]
First padded token sequence:
['the', 'Ġmovie', 'ĠThe', 'ĠIn', '-L', 'aws', 'Ġnot', 'Ġexactly', 'Ġa', 'Ġholiday', 'Ġmovie', 'Ġbut', 'Ġfunny', 'Ġand', 'Ġgood', '!', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']


## 6. Run a forward pass and capture hidden states

For a decoder with `L` layers, Hugging Face returns `L + 1` hidden-state tensors: the embedding output followed by one tensor for each decoder layer. Every tensor has shape `[batch, sequence_length, hidden_size]`.

`use_cache=False` avoids allocating the key/value cache because this pass analyzes complete sentences rather than generating tokens.

In [10]:
model_inputs = {name: tensor.to(device) for name, tensor in encoded.items()}

with torch.inference_mode():
    outputs = model(
        **model_inputs,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

hidden_states = outputs.hidden_states
print("Logits shape:", tuple(outputs.logits.shape))
print("Number of hidden-state tensors:", len(hidden_states))
print("Embedding output shape:", tuple(hidden_states[0].shape))
print("First layer output shape:", tuple(hidden_states[1].shape))
print("Final layer output shape:", tuple(hidden_states[-1].shape))
print("Expected hidden-state count:", config.num_hidden_layers + 1)
assert len(hidden_states) == config.num_hidden_layers + 1
assert hidden_states[-1].shape[-1] == config.hidden_size

Logits shape: (12, 22, 151936)
Number of hidden-state tensors: 25
Embedding output shape: (12, 22, 896)
First layer output shape: (12, 22, 896)
Final layer output shape: (12, 22, 896)
Expected hidden-state count: 25


## 7. Convert token activations into sentence representations

We will compare two transparent baselines:

- **Mean pooling:** average all non-padding token activations. This includes information distributed across the sentence.
- **Final-token pooling:** select the final non-padding token. In a causal model, this token can attend to the full preceding sentence.

Neither is assumed to be correct. The later extraction experiment must compare them on held-out data.

In [11]:
def mean_pool(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    expanded_mask = attention_mask.unsqueeze(-1).to(hidden_state.dtype)
    summed = (hidden_state * expanded_mask).sum(dim=1)
    counts = expanded_mask.sum(dim=1).clamp_min(1)
    return summed / counts


def final_token_pool(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    final_positions = attention_mask.sum(dim=1) - 1
    batch_positions = torch.arange(hidden_state.shape[0], device=hidden_state.device)
    return hidden_state[batch_positions, final_positions]


selected_layer = config.num_hidden_layers // 2
selected_hidden = hidden_states[selected_layer + 1]
mean_representations = mean_pool(selected_hidden, model_inputs["attention_mask"])
final_representations = final_token_pool(selected_hidden, model_inputs["attention_mask"])

print("Selected decoder layer:", selected_layer)
print("Token activations:", tuple(selected_hidden.shape))
print("Mean-pooled representations:", tuple(mean_representations.shape))
print("Final-token representations:", tuple(final_representations.shape))

Selected decoder layer: 12
Token activations: (12, 22, 896)
Mean-pooled representations: (12, 896)
Final-token representations: (12, 896)


### A tiny shape-check for paired differences

The following calculation demonstrates the eventual extraction operation at several layers. Six examples are far too few to support a research conclusion; the numbers are only a mechanical sanity check.

In [ ]:
number_of_pairs = len(example_pairs)
layer_rows = []
layers_to_inspect = sorted(
    {0, 1, config.num_hidden_layers // 4, config.num_hidden_layers // 2,
     3 * config.num_hidden_layers // 4, config.num_hidden_layers}
)

for hidden_state_index in layers_to_inspect:
    pooled = mean_pool(
        hidden_states[hidden_state_index], model_inputs["attention_mask"]
    ).float()
    informal_representations = pooled[:number_of_pairs]
    formal_representations = pooled[number_of_pairs:]
    deltas = formal_representations - informal_representations  ## vector for formality, 
    mean_delta = deltas.mean(dim=0)
    alignments = F.cosine_similarity(deltas, mean_delta.unsqueeze(0), dim=1)
    layer_rows.append(
        {
            "hidden_state_index": hidden_state_index,
            "meaning": (
                "embedding output" if hidden_state_index == 0
                else "final normalized output"
                if hidden_state_index == config.num_hidden_layers
                else f"after decoder layer {hidden_state_index - 1}"
            ),
            "mean_delta_l2_norm": mean_delta.norm().item(),
            "mean_pair_alignment": alignments.mean().item(),
        }
    )

pd.DataFrame(layer_rows).round(4)

,hidden_state_index,meaning,mean_delta_l2_norm,mean_pair_alignment
0,0,embedding output,0.0501,0.4727
1,1,after decoder layer 0,0.6794,0.5047
2,6,after decoder layer 5,37.8138,0.8858
3,12,after decoder layer 11,38.0005,0.8765
4,18,after decoder layer 17,38.3168,0.8442
5,24,final normalized output,46.2541,0.5564


## 8. Observe a decoder layer with a forward hook

A forward hook receives a module's input and output whenever that module runs. This hook only records the output; it does not modify the activation. Later, a steering hook would return a changed tensor of the same shape.

For decoder layer `k`, the hooked residual output should correspond to `hidden_states[k + 1]`. We use a middle layer because some architectures apply a final normalization after the last decoder layer.

In [13]:
target_layer_index = config.num_hidden_layers // 2
captured = {}


def capture_layer_output(module, module_inputs, module_output):
    residual_output = module_output[0] if isinstance(module_output, tuple) else module_output
    captured["residual_output"] = residual_output.detach()


hook_handle = decoder_layers[target_layer_index].register_forward_hook(
    capture_layer_output
)
try:
    with torch.inference_mode():
        hooked_outputs = model(
            **model_inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
finally:
    hook_handle.remove()

hooked_residual = captured["residual_output"]
returned_residual = hooked_outputs.hidden_states[target_layer_index + 1]
maximum_difference = (hooked_residual - returned_residual).abs().max().item()

print("Hooked module:", type(decoder_layers[target_layer_index]).__name__)
print("Target layer index:", target_layer_index)
print("Hooked output shape:", tuple(hooked_residual.shape))
print("Returned hidden-state shape:", tuple(returned_residual.shape))
print("Maximum absolute difference:", maximum_difference)
assert hooked_residual.shape == returned_residual.shape

Hooked module: Qwen2DecoderLayer
Target layer index: 12
Hooked output shape: (12, 22, 896)
Returned hidden-state shape: (12, 22, 896)
Maximum absolute difference: 0.0


## 9. Establish the unsteered generation baseline

This uses the model's chat template and greedy decoding. The output is a baseline, not evidence about hidden-state geometry. A 0.5B model may produce imperfect rewrites; that is acceptable for the anatomy stage.

In [14]:
sentence_to_rewrite = example_pairs.loc[2, "informal"]
messages = [
    {
        "role": "system",
        "content": (
            "Rewrite the user's sentence in formal English while preserving its meaning. "
            "Return only the rewritten sentence."
        ),
    },
    {"role": "user", "content": sentence_to_rewrite},
]

prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
generation_inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

with torch.inference_mode():
    generated_ids = model.generate(
        **generation_inputs,
        max_new_tokens=48,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_token_ids = generated_ids[0, generation_inputs["input_ids"].shape[1]:]
generated_rewrite = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

print("Informal input:", sentence_to_rewrite)
print("GYAFC reference:", example_pairs.loc[2, "formal"])
print("Model rewrite:", generated_rewrite)

Informal input: of corse i be wachin it evry day, my fav charachter is Inuasha
GYAFC reference: I watch it everyday, my favorite charachter is Inuasha.
Model rewrite: Of course, I always keep Inuasha as my favorite character, regardless of the day.


## 10. What we should know before moving on

After running the notebook, verify that you can answer these questions from the displayed values:

1. What IDs and tokens does the tokenizer produce?
2. What do batch size, sequence length, and hidden size mean in `[batch, tokens, hidden]`?
3. Why are there `number_of_layers + 1` hidden-state tensors?
4. How does the attention mask prevent padding from affecting mean pooling?
5. Why might mean pooling and final-token pooling encode different information?
6. Which module produces the residual activation that a steering hook would change?
7. Can the unmodified model follow a formal-rewrite instruction at all?

### Next experiment

Once these mechanics are clear, build a separate activation-extraction notebook. Start with 500–1,000 aligned pairs, compute paired differences at every layer for both pooling methods, and evaluate held-out ordering, direction consistency, split stability, and cross-domain transfer. Do not modify activations until that geometry is validated.

## 11. Results and interpretation

### Runtime and device

The notebook was run with Python 3.11.0 from `/usr/local/bin/python3`, PyTorch 2.12.1, and Transformers 5.14.1. MPS was available, and the model ran on `mps:0` with float16 parameters. This confirms that the Mac can run the model with hardware acceleration. The parameter tensors occupy about **0.92 GiB**, leaving substantial memory for activations and moderate batch sizes on a 24 GB machine.

> **Reproducibility note:** this run used the system Python rather than the project's `SecondThought/.venv` environment. The results are valid, but the larger extraction experiment should use one fixed environment so library-version changes do not become an experimental variable.

### Model loading and architecture

`Qwen/Qwen2.5-0.5B-Instruct` loaded successfully with **494,032,768 parameters**. It has 24 decoder layers, hidden size 896, 14 query-attention heads, 2 key/value heads, and a vocabulary of 151,936 tokens. The decoder stack is available as `model.model.layers`, each element is a `Qwen2DecoderLayer`, the stack ends with `Qwen2RMSNorm`, and logits are produced by a linear language-model head. These observations identify the precise modules and residual-stream width needed for later extraction and intervention.

The Hugging Face warning about unauthenticated requests affected only download rate limits; it did not affect the loaded model or calculations.

### GYAFC examples and tokenization

Six paired examples were loaded, three from each domain. They already reveal an important confound: the formal rewrites combine spelling correction, contraction changes, punctuation, lexical replacement, deletion, and some semantic rewriting. A paired activation difference therefore cannot automatically be interpreted as pure formality.

The first informal sentence becomes 16 subword tokens. For example, `In-Laws` is split into `ĠIn`, `-L`, and `aws`, while the exclamation mark is its own token. The `Ġ` marker indicates that a token begins after a space. This confirms that informal and formal sentences cannot be aligned reliably by token position. A sentence-level pooling operation is needed before subtracting paired representations.

The batch contains 12 sentences and has shape `[12, 22]`: 12 sequences padded to the longest sequence length of 22 tokens. Actual lengths range from 9 to 22 tokens. `<|endoftext|>` fills padded positions, while the attention mask distinguishes padding from real tokens. The mask must be used during mean pooling so padding does not contaminate a representation.

### Hidden states and pooling

The forward pass returns logits of shape `[12, 22, 151936]` and **25 hidden-state tensors**. This is correct for a 24-layer decoder: index 0 is the embedding output, followed by the returned representation at each layer. Every hidden-state tensor has shape `[12, 22, 896]`, corresponding to `[batch, token position, hidden dimension]`.

At decoder layer 12, both pooling methods reduce `[12, 22, 896]` token activations to `[12, 896]` sentence representations. Mean pooling summarizes all non-padding positions. Final-token pooling selects the last real token, whose causal representation can incorporate all preceding tokens. We have not yet established which pooling method captures formality more reliably; that requires held-out comparison.

### Preliminary paired-direction signal

The six-example mean-pooled diagnostic produced the following pattern:

| Hidden-state location | Mean delta norm | Mean pair alignment |
|---|---:|---:|
| Embedding output | 0.0501 | 0.4727 |
| After decoder layer 0 | 0.6794 | 0.5047 |
| After decoder layer 5 | 37.8138 | **0.8858** |
| After decoder layer 11 | 38.0005 | **0.8765** |
| After decoder layer 17 | 38.3168 | **0.8442** |
| Final normalized output | 46.2541 | 0.5564 |

There is a **preliminary sense of a shared direction in the intermediate hidden layers**. Pair differences are much more aligned with their mean direction after layers 5, 11, and 17 than at the embedding, first-layer, or final-output stages. This is the pattern we hoped might exist: intermediate layers may organize the common properties of the rewrites more coherently than raw token embeddings do. The decline at the final output is also plausible because the last representation is optimized for next-token prediction rather than sentence-level style geometry.

However, this is **not yet evidence that a generalizable formality direction exists** for four reasons:

1. The direction was estimated from the same six pairs whose alignment was measured, so the result has in-sample bias.
2. Six pairs are far too few to characterize a 896-dimensional space.
3. No shuffled-label, random-direction, or held-out baseline was used.
4. The deltas contain spelling correction, semantic edits, length changes, and domain content in addition to formality.

The increasing delta norms are not by themselves evidence of stronger formality encoding because activation scale changes substantially across layers. Cosine alignment and held-out projection behavior are more informative than raw norm when comparing layers.

### Hook verification

The read-only hook on decoder layer 12 captured a tensor with shape `[12, 22, 896]`. Its maximum absolute difference from `hidden_states[13]` was exactly **0.0**. We therefore know where to observe—and eventually modify—the residual activation. This validates the intervention location mechanically, but no activation was changed in this notebook.

### Generation baseline

The unsteered model corrected spelling and produced more polished language, but it changed `I watch it every day` into `I always keep Inuasha as my favorite character`. The output is fluent and more formal, yet it does not fully preserve the original meaning. This demonstrates why semantic preservation must be measured independently from formality. It also establishes that successful-looking prose is not sufficient evidence that steering worked correctly.

### Conclusion and decision

The notebook provides an encouraging hypothesis: **layers roughly 5–17 may contain a more coherent formalization-related direction than the input or final layer**. We should now test that hypothesis rather than steer with the six-example vector.

The next experiment should use 500–1,000 paired examples with a pair-level extraction/held-out split. At every layer, compare mean pooling and final-token pooling; estimate the direction only on the extraction subset; measure held-out formal-over-informal ordering, cosine consistency, bootstrap stability, and cross-domain transfer; and compare against shuffled-label directions. Exact copies should be retained as controls. Only a layer that passes these held-out tests should be used for causal activation steering.

In [ ]:
# Large-scale, held-out formality-direction experiment.
#
# This cell deliberately keeps extraction and evaluation separate:
#   1. sample non-identical GYAFC pairs independently within each domain;
#   2. collect formal-minus-informal deltas at every returned depth;
#   3. estimate directions using only the extraction split;
#   4. evaluate those frozen directions using only held-out pairs.
#
# Default workload: 8,000 extraction + 2,000 held-out pairs = 20,000
# sentence forward passes. Reduce the per-domain values for a quicker test.

from IPython.display import display
import gc
import math
import time
import numpy as np

EXTRACTION_PAIRS_PER_DOMAIN = 4_000
HELDOUT_PAIRS_PER_DOMAIN = 1_000

# delta * formal_sentences (for hidden )
EXTRACTION_BATCH_SIZE = 32  # Number of pairs; the model sees twice this many sentences.
EXTRACTION_MAX_LENGTH = 128
NUMBER_OF_SHUFFLED_CONTROLS = 3
EXPERIMENT_SEED = 42

experiment_rng = np.random.default_rng(EXPERIMENT_SEED)
tokenizer.padding_side = "right"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


def load_training_pairs_for_experiment() -> pd.DataFrame:
    frames = []
    for domain, directory in DOMAIN_DIRECTORIES.items():
        train_directory = DATA_ROOT / directory / "train"
        informal = (train_directory / "informal").read_text(encoding="utf-8").splitlines()
        formal = (train_directory / "formal").read_text(encoding="utf-8").splitlines()
        assert len(informal) == len(formal), f"Unaligned training files for {domain}"
        frames.append(
            pd.DataFrame(
                {
                    "domain": domain,
                    "domain_row": np.arange(len(informal)),
                    "informal": informal,
                    "formal": formal,
                }
            )
        )

    frame = pd.concat(frames, ignore_index=True)
    informal_normalized = frame["informal"].str.strip().str.lower().str.replace(
        r"\s+", " ", regex=True
    )
    formal_normalized = frame["formal"].str.strip().str.lower().str.replace(
        r"\s+", " ", regex=True
    )
    frame["exact_copy"] = informal_normalized == formal_normalized
    return frame


all_training_pairs = load_training_pairs_for_experiment()
eligible_pairs = all_training_pairs.loc[~all_training_pairs["exact_copy"]].copy()

extraction_parts = []
heldout_parts = []
for domain in DOMAIN_DIRECTORIES:
    domain_frame = eligible_pairs.loc[eligible_pairs["domain"] == domain]
    required = EXTRACTION_PAIRS_PER_DOMAIN + HELDOUT_PAIRS_PER_DOMAIN
    if len(domain_frame) < required:
        raise ValueError(
            f"Requested {required:,} non-copy pairs from {domain}, "
            f"but only {len(domain_frame):,} are available."
        )
    shuffled_positions = experiment_rng.permutation(len(domain_frame))
    extraction_positions = shuffled_positions[:EXTRACTION_PAIRS_PER_DOMAIN]
    heldout_positions = shuffled_positions[
        EXTRACTION_PAIRS_PER_DOMAIN:required
    ]
    extraction_parts.append(domain_frame.iloc[extraction_positions])
    heldout_parts.append(domain_frame.iloc[heldout_positions])

extraction_pairs = (
    pd.concat(extraction_parts)
    .sample(frac=1, random_state=EXPERIMENT_SEED)
    .reset_index(drop=True)
)
heldout_pairs = (
    pd.concat(heldout_parts)
    .sample(frac=1, random_state=EXPERIMENT_SEED + 1)
    .reset_index(drop=True)
)

assert not set(
    zip(extraction_pairs["domain"], extraction_pairs["domain_row"])
) & set(zip(heldout_pairs["domain"], heldout_pairs["domain_row"]))

split_summary = pd.DataFrame(
    {
        "extraction": extraction_pairs.groupby("domain").size(),
        "heldout": heldout_pairs.groupby("domain").size(),
    }
)
print("Experiment split (exact copies excluded):")
display(split_summary)


def collect_all_layer_deltas(
    frame: pd.DataFrame,
    batch_size: int,
    max_length: int,
    split_name: str,
) -> tuple[torch.Tensor, int]:
    """Return float16 deltas with shape [pooling, pairs, depths, hidden]."""
    number_of_pairs = len(frame)
    number_of_depths = config.num_hidden_layers + 1
    deltas = torch.empty(
        (2, number_of_pairs, number_of_depths, config.hidden_size),
        dtype=torch.float16,
        device="cpu",
    )
    truncated_sentence_count = 0
    started = time.perf_counter()
    total_batches = math.ceil(number_of_pairs / batch_size)

    for batch_number, start in enumerate(range(0, number_of_pairs, batch_size), start=1):
        stop = min(start + batch_size, number_of_pairs)
        batch = frame.iloc[start:stop]
        pair_count = len(batch)
        texts = batch["informal"].tolist() + batch["formal"].tolist()

        untruncated_ids = tokenizer(
            texts,
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )["input_ids"]
        truncated_sentence_count += sum(
            len(token_ids) > max_length for token_ids in untruncated_ids
        )

        batch_encoding = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        batch_inputs = {
            name: tensor.to(device) for name, tensor in batch_encoding.items()
        }

        # Call the base decoder rather than the causal-LM wrapper. We need hidden
        # states, not the very large [batch, tokens, vocabulary] logits tensor.
        with torch.inference_mode():
            decoder_outputs = model.model(
                **batch_inputs,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )

        attention_mask = batch_inputs["attention_mask"]
        for depth_index, layer_hidden in enumerate(decoder_outputs.hidden_states):
            mean_vectors = mean_pool(layer_hidden, attention_mask)
            final_vectors = final_token_pool(layer_hidden, attention_mask)

            mean_delta = mean_vectors[pair_count:] - mean_vectors[:pair_count]
            final_delta = final_vectors[pair_count:] - final_vectors[:pair_count]
            deltas[0, start:stop, depth_index] = mean_delta.to(
                device="cpu", dtype=torch.float16
            )
            deltas[1, start:stop, depth_index] = final_delta.to(
                device="cpu", dtype=torch.float16
            )

        del decoder_outputs, batch_inputs, batch_encoding
        if batch_number == 1 or batch_number % 20 == 0 or batch_number == total_batches:
            elapsed = time.perf_counter() - started
            print(
                f"{split_name}: batch {batch_number}/{total_batches} "
                f"({stop:,}/{number_of_pairs:,} pairs, {elapsed:.1f}s)"
            )

    return deltas, truncated_sentence_count


extraction_deltas, extraction_truncated = collect_all_layer_deltas(
    extraction_pairs,
    batch_size=EXTRACTION_BATCH_SIZE,
    max_length=EXTRACTION_MAX_LENGTH,
    split_name="extraction",
)
heldout_deltas, heldout_truncated = collect_all_layer_deltas(
    heldout_pairs,
    batch_size=EXTRACTION_BATCH_SIZE,
    max_length=EXTRACTION_MAX_LENGTH,
    split_name="heldout",
)

print(
    "Stored delta tensor shapes:",
    tuple(extraction_deltas.shape),
    tuple(heldout_deltas.shape),
)
print(
    f"Sentences truncated at {EXTRACTION_MAX_LENGTH} tokens: "
    f"{extraction_truncated + heldout_truncated:,}"
)

# Create two disjoint, domain-balanced extraction halves for stability checks.
extraction_domains = extraction_pairs["domain"].to_numpy()
heldout_domains = heldout_pairs["domain"].to_numpy()
half_assignment = np.empty(len(extraction_pairs), dtype=np.int8)
for domain in DOMAIN_DIRECTORIES:
    positions = np.flatnonzero(extraction_domains == domain)
    positions = experiment_rng.permutation(positions)
    midpoint = len(positions) // 2
    half_assignment[positions[:midpoint]] = 0
    half_assignment[positions[midpoint:]] = 1

# Randomly reversing formal/informal labels destroys a consistent direction
# while retaining the same activation differences and magnitudes.
shuffled_signs = torch.from_numpy(
    experiment_rng.choice(
        np.array([-1.0, 1.0], dtype=np.float32),
        size=(NUMBER_OF_SHUFFLED_CONTROLS, len(extraction_pairs)),
    )
)

pooling_names = ["mean", "final_token"]
domain_names = list(DOMAIN_DIRECTORIES)
result_rows = []

for pooling_index, pooling_name in enumerate(pooling_names):
    for depth_index in range(config.num_hidden_layers + 1):
        extraction_layer_deltas = extraction_deltas[
            pooling_index, :, depth_index
        ].float()
        heldout_layer_deltas = heldout_deltas[
            pooling_index, :, depth_index
        ].float()

        mean_delta = extraction_layer_deltas.mean(dim=0)
        direction = mean_delta / mean_delta.norm().clamp_min(1e-12)

        projection_gaps = heldout_layer_deltas @ direction
        heldout_norms = heldout_layer_deltas.norm(dim=1)
        nonzero = heldout_norms > 0
        cosine_alignments = projection_gaps[nonzero] / heldout_norms[nonzero]

        ordering_accuracy = (projection_gaps > 0).float().mean().item()
        accuracy_se = math.sqrt(
            ordering_accuracy * (1 - ordering_accuracy) / len(projection_gaps)
        )
        gap_std = projection_gaps.std(unbiased=True).item()

        half_directions = []
        for half in (0, 1):
            half_mean = extraction_layer_deltas[half_assignment == half].mean(dim=0)
            half_directions.append(half_mean / half_mean.norm().clamp_min(1e-12))
        split_half_cosine = F.cosine_similarity(
            half_directions[0], half_directions[1], dim=0
        ).item()

        domain_directions_for_layer = {}
        for domain in domain_names:
            domain_mean = extraction_layer_deltas[
                extraction_domains == domain
            ].mean(dim=0)
            domain_directions_for_layer[domain] = (
                domain_mean / domain_mean.norm().clamp_min(1e-12)
            )
        domain_direction_cosine = F.cosine_similarity(
            domain_directions_for_layer[domain_names[0]],
            domain_directions_for_layer[domain_names[1]],
            dim=0,
        ).item()

        cross_domain_accuracies = {}
        for source_domain, target_domain in (
            (domain_names[0], domain_names[1]),
            (domain_names[1], domain_names[0]),
        ):
            target_deltas = heldout_layer_deltas[heldout_domains == target_domain]
            target_gaps = target_deltas @ domain_directions_for_layer[source_domain]
            cross_domain_accuracies[
                f"{source_domain}_to_{target_domain}_accuracy"
            ] = (target_gaps > 0).float().mean().item()

        shuffled_means = (
            shuffled_signs @ extraction_layer_deltas
        ) / len(extraction_layer_deltas)
        shuffled_directions = shuffled_means / shuffled_means.norm(
            dim=1, keepdim=True
        ).clamp_min(1e-12)
        shuffled_gaps = shuffled_directions @ heldout_layer_deltas.T
        shuffled_accuracy = (
            (shuffled_gaps > 0).float().mean(dim=1).mean().item()
        )

        if depth_index == 0:
            location = "embedding output"
        elif depth_index == config.num_hidden_layers:
            location = "final normalized output"
        else:
            location = f"after decoder layer {depth_index - 1}"

        result_rows.append(
            {
                "pooling": pooling_name,
                "hidden_state_index": depth_index,
                "location": location,
                "mean_delta_norm": mean_delta.norm().item(),
                "heldout_ordering_accuracy": ordering_accuracy,
                "accuracy_95ci_low": max(0.0, ordering_accuracy - 1.96 * accuracy_se),
                "accuracy_95ci_high": min(1.0, ordering_accuracy + 1.96 * accuracy_se),
                "heldout_tie_rate": torch.isclose(
                    projection_gaps, torch.tensor(0.0)
                ).float().mean().item(),
                "projection_effect_size": (
                    projection_gaps.mean().item() / gap_std if gap_std > 0 else np.nan
                ),
                "mean_heldout_delta_cosine": cosine_alignments.mean().item(),
                "median_heldout_delta_cosine": cosine_alignments.median().item(),
                "split_half_direction_cosine": split_half_cosine,
                "domain_direction_cosine": domain_direction_cosine,
                "shuffled_label_accuracy": shuffled_accuracy,
                "accuracy_above_shuffle": ordering_accuracy - shuffled_accuracy,
                **cross_domain_accuracies,
            }
        )

layer_results = pd.DataFrame(result_rows)
metric_columns = [
    "pooling",
    "hidden_state_index",
    "location",
    "mean_delta_norm",
    "heldout_ordering_accuracy",
    "accuracy_95ci_low",
    "accuracy_95ci_high",
    "heldout_tie_rate",
    "projection_effect_size",
    "mean_heldout_delta_cosine",
    "median_heldout_delta_cosine",
    "split_half_direction_cosine",
    "domain_direction_cosine",
    "shuffled_label_accuracy",
    "accuracy_above_shuffle",
    "entertainment_music_to_family_relationships_accuracy",
    "family_relationships_to_entertainment_music_accuracy",
]

print("All layers and both pooling methods:")
display(layer_results[metric_columns].round(4))

print("Strongest held-out candidates:")
top_layer_candidates = layer_results.sort_values(
    ["heldout_ordering_accuracy", "split_half_direction_cosine"],
    ascending=False,
).head(12)
display(top_layer_candidates[metric_columns].round(4))

print("Best candidate within each pooling method:")
best_by_pooling = layer_results.loc[
    layer_results.groupby("pooling")["heldout_ordering_accuracy"].idxmax()
]
display(best_by_pooling[metric_columns].round(4))

print(
    "Interpret a layer as credible only when held-out accuracy is clearly above "
    "both 0.50 and shuffled-label accuracy, split-half and domain-direction "
    "cosines are positive and stable, and cross-domain accuracies remain above chance."
)

# Keep layer_results and the delta tensors available for follow-up analysis.
# To reclaim roughly 0.9 GiB after inspection, run:
# del extraction_deltas, heldout_deltas; gc.collect(); torch.mps.empty_cache()


Experiment split (exact copies excluded):


,extraction,heldout
domain,,
entertainment_music,4000,1000
family_relationships,4000,1000


extraction: batch 1/250 (32/8,000 pairs, 0.7s)
extraction: batch 20/250 (640/8,000 pairs, 7.5s)
extraction: batch 40/250 (1,280/8,000 pairs, 14.9s)
extraction: batch 60/250 (1,920/8,000 pairs, 22.1s)
extraction: batch 80/250 (2,560/8,000 pairs, 30.0s)
extraction: batch 100/250 (3,200/8,000 pairs, 37.5s)
extraction: batch 120/250 (3,840/8,000 pairs, 45.2s)
extraction: batch 140/250 (4,480/8,000 pairs, 51.9s)
extraction: batch 160/250 (5,120/8,000 pairs, 59.3s)
extraction: batch 180/250 (5,760/8,000 pairs, 66.7s)
extraction: batch 200/250 (6,400/8,000 pairs, 73.7s)
extraction: batch 220/250 (7,040/8,000 pairs, 80.6s)
extraction: batch 240/250 (7,680/8,000 pairs, 87.4s)
extraction: batch 250/250 (8,000/8,000 pairs, 90.8s)
heldout: batch 1/63 (32/2,000 pairs, 0.4s)
heldout: batch 20/63 (640/2,000 pairs, 7.2s)
heldout: batch 40/63 (1,280/2,000 pairs, 14.2s)
heldout: batch 60/63 (1,920/2,000 pairs, 21.3s)
heldout: batch 63/63 (2,000/2,000 pairs, 22.4s)
Stored delta tensor shapes: (2, 8000, 2

,pooling,hidden_state_index,location,mean_delta_norm,heldout_ordering_accuracy,accuracy_95ci_low,accuracy_95ci_high,heldout_tie_rate,projection_effect_size,mean_heldout_delta_cosine,median_heldout_delta_cosine,split_half_direction_cosine,domain_direction_cosine,shuffled_label_accuracy,accuracy_above_shuffle,entertainment_music_to_family_relationships_accuracy,family_relationships_to_entertainment_music_accuracy
0,mean,0,embedding output,0.0323,0.8315,0.8151,0.8479,0.000,0.8384,0.2658,0.2949,0.9964,0.9735,0.4560,0.3755,0.807,0.859
1,mean,1,after decoder layer 0,0.4030,0.9520,0.9426,0.9614,0.000,1.3050,0.2842,0.2994,0.9972,0.9710,0.4158,0.5362,0.941,0.964
2,mean,2,after decoder layer 1,0.6183,0.9595,0.9509,0.9681,0.000,1.5136,0.3007,0.3152,0.9975,0.9707,0.4608,0.4987,0.953,0.970
3,mean,3,after decoder layer 2,3.0609,0.5650,0.5433,0.5867,0.000,0.1499,0.1075,0.3895,0.9996,0.9975,0.5077,0.0573,0.582,0.548
4,mean,4,after decoder layer 3,3.6554,0.5220,0.5001,0.5439,0.000,0.0740,0.0304,0.0499,0.9991,0.9934,0.4977,0.0243,0.536,0.512
5,mean,5,after decoder layer 4,3.6900,0.5235,0.5016,0.5454,0.000,0.0765,0.0381,0.0781,0.9987,0.9908,0.4992,0.0243,0.537,0.508
6,mean,6,after decoder layer 5,3.7521,0.5255,0.5036,0.5474,0.000,0.0773,0.0402,0.0854,0.9987,0.9906,0.4983,0.0272,0.538,0.510
7,mean,7,after decoder layer 6,3.7759,0.5285,0.5066,0.5504,0.000,0.0785,0.0430,0.0960,0.9986,0.9899,0.4995,0.0290,0.542,0.514
8,mean,8,after decoder layer 7,3.7568,0.5275,0.5056,0.5494,0.000,0.0780,0.0423,0.0851,0.9985,0.9900,0.4995,0.0280,0.545,0.512
9,mean,9,after decoder layer 8,3.8110,0.5300,0.5081,0.5519,0.000,0.0799,0.0453,0.0888,0.9984,0.9901,0.4992,0.0308,0.545,0.514


Strongest held-out candidates:


,pooling,hidden_state_index,location,mean_delta_norm,heldout_ordering_accuracy,accuracy_95ci_low,accuracy_95ci_high,heldout_tie_rate,projection_effect_size,mean_heldout_delta_cosine,median_heldout_delta_cosine,split_half_direction_cosine,domain_direction_cosine,shuffled_label_accuracy,accuracy_above_shuffle,entertainment_music_to_family_relationships_accuracy,family_relationships_to_entertainment_music_accuracy
22,mean,22,after decoder layer 21,5.6086,0.9665,0.9586,0.9744,0.0,1.5870,0.3627,0.3733,0.9983,0.9747,0.6027,0.3638,0.965,0.967
23,mean,23,after decoder layer 22,6.6690,0.9595,0.9509,0.9681,0.0,1.5690,0.3713,0.3849,0.9984,0.9753,0.5612,0.3983,0.959,0.961
2,mean,2,after decoder layer 1,0.6183,0.9595,0.9509,0.9681,0.0,1.5136,0.3007,0.3152,0.9975,0.9707,0.4608,0.4987,0.953,0.970
30,final_token,5,after decoder layer 4,2.5102,0.9520,0.9426,0.9614,0.0,1.1176,0.3036,0.3266,0.9982,0.9529,0.4760,0.4760,0.944,0.961
1,mean,1,after decoder layer 0,0.4030,0.9520,0.9426,0.9614,0.0,1.3050,0.2842,0.2994,0.9972,0.9710,0.4158,0.5362,0.941,0.964
31,final_token,6,after decoder layer 5,2.8996,0.9510,0.9415,0.9605,0.0,1.1253,0.3166,0.3439,0.9984,0.9548,0.4648,0.4862,0.941,0.955
45,final_token,20,after decoder layer 19,7.2880,0.9495,0.9399,0.9591,0.0,1.2400,0.3523,0.3906,0.9988,0.9522,0.4508,0.4987,0.930,0.959
43,final_token,18,after decoder layer 17,6.0787,0.9455,0.9356,0.9554,0.0,1.1290,0.3483,0.3845,0.9988,0.9620,0.4498,0.4957,0.935,0.955
32,final_token,7,after decoder layer 6,3.0331,0.9455,0.9356,0.9554,0.0,1.0574,0.3131,0.3387,0.9985,0.9531,0.4665,0.4790,0.941,0.947
44,final_token,19,after decoder layer 18,6.6583,0.9435,0.9334,0.9536,0.0,1.1480,0.3520,0.3918,0.9988,0.9605,0.4473,0.4962,0.933,0.954


Best candidate within each pooling method:


,pooling,hidden_state_index,location,mean_delta_norm,heldout_ordering_accuracy,accuracy_95ci_low,accuracy_95ci_high,heldout_tie_rate,projection_effect_size,mean_heldout_delta_cosine,median_heldout_delta_cosine,split_half_direction_cosine,domain_direction_cosine,shuffled_label_accuracy,accuracy_above_shuffle,entertainment_music_to_family_relationships_accuracy,family_relationships_to_entertainment_music_accuracy
30,final_token,5,after decoder layer 4,2.5102,0.9520,0.9426,0.9614,0.0,1.1176,0.3036,0.3266,0.9982,0.9529,0.4760,0.4760,0.944,0.961
22,mean,22,after decoder layer 21,5.6086,0.9665,0.9586,0.9744,0.0,1.5870,0.3627,0.3733,0.9983,0.9747,0.6027,0.3638,0.965,0.967


Interpret a layer as credible only when held-out accuracy is clearly above both 0.50 and shuffled-label accuracy, split-half and domain-direction cosines are positive and stable, and cross-domain accuracies remain above chance.


## 12. Large-scale direction experiment report

### Experiment integrity

The experiment used **8,000 extraction pairs** and **2,000 completely held-out pairs**, balanced equally between Entertainment & Music and Family & Relationships. Exact-copy pairs were excluded. The model processed 20,000 sentences in approximately 113 seconds on MPS, and **no sentence was truncated** at the 128-token limit. The extraction and held-out tensors have shape [2 pooling methods, pairs, 25 representation depths, 896 hidden dimensions]. This is large enough to replace the earlier six-example observation with a meaningful held-out geometric test.

### What each metric means

- **mean_delta_norm** is the length of the extraction direction before normalization. A larger value does not necessarily mean a better direction because activation scale changes across layers. Do not rank layers using this value alone.
- **heldout_ordering_accuracy** is the fraction of unseen pairs satisfying (h_formal - h_informal) · v_formal > 0. Chance is approximately 0.50. This is the clearest measure of whether the direction generalizes.
- **accuracy_95ci_low/high** is an approximate 95% interval around held-out accuracy. An interval well above 0.50 indicates that the result is unlikely to be sampling noise from the 2,000 held-out pairs. It does not include uncertainty from choosing the extraction sample or layer.
- **heldout_tie_rate** is the fraction with a projection gap of zero. Ties count as failures in ordering accuracy. A large tie rate means the representation cannot distinguish many pairs along that direction.
- **projection_effect_size** is the mean held-out projection gap divided by its standard deviation. Values near zero indicate weak separation; values around or above 1 indicate strong separation relative to pair-to-pair variation.
- **mean/median_heldout_delta_cosine** measures how closely each unseen pair difference points toward the aggregate direction. Zero means no directional alignment, positive values support a shared direction, and 1 would mean every pair moves exactly the same way. Moderate values are expected because formalization contains several transformations.
- **split_half_direction_cosine** compares directions estimated from two disjoint, domain-balanced extraction halves. Values near 1 indicate a highly reproducible aggregate direction.
- **domain_direction_cosine** compares directions estimated independently from the two GYAFC domains. Values near 1 indicate that the direction is not primarily topic-specific.
- **shuffled_label_accuracy** is accuracy after randomly reversing formal/informal labels during direction estimation. It is an empirical null control. Only three shuffled controls were used, so this estimate is noisy.
- **accuracy_above_shuffle** is real held-out accuracy minus shuffled-label accuracy.
- **Cross-domain accuracy** learns the direction in one domain and evaluates it in the other. Strong performance in both directions supports a shared formalization signal rather than domain vocabulary alone.

### Mean-pooling behavior across depth

Mean pooling shows a striking non-monotonic trajectory:

| Representation | Held-out accuracy | Effect size | Interpretation |
|---|---:|---:|---|
| Embeddings | 83.15% | 0.838 | Surface token differences already provide a substantial signal. |
| After layer 0 | 95.20% | 1.305 | The first contextual layer strongly organizes the paired distinction. |
| After layer 1 | 95.95% | 1.514 | Very strong early-layer separation. |
| After layer 2 | 56.50% | 0.150 | Separation drops sharply. |
| After layers 3–16 | roughly 52–54% | roughly 0.07–0.10 | The mean-pooled linear signal nearly disappears. |
| After layers 17–20 | 55.50–61.20% | 0.128–0.231 | The signal gradually re-emerges. |
| After layer 21 | **96.65%** | **1.587** | Strongest overall held-out candidate. |
| After layer 22 | 95.95% | 1.569 | Nearly as strong as layer 21. |
| Final normalized output | 93.10% | 1.389 | Strong, but weaker than the preceding two depths. |

The best candidate is hidden-state index 22, corresponding to the output after decoder layer 21. Its 95% accuracy interval is **95.86–97.44%**. Mean and median held-out delta cosines are **0.3627** and **0.3733**: individual rewrites are not parallel, but they share a clear positive component. Split-half cosine is **0.9983**, domain-direction cosine is **0.9747**, and cross-domain accuracies are **96.5%** and **96.7%**. These values jointly indicate a stable and transferable direction.

The middle-layer collapse is important. Direction stability remains extremely high even where ordering accuracy is near chance. Stability means that an average difference is reproducible; it does **not** mean that the direction usefully orders individual pairs. Stability and held-out separation must be interpreted together. The collapse may mean that information becomes distributed in a way simple mean pooling cannot recover, followed by late-layer reorganization for prediction.

### Final-token behavior across depth

- The embedding result is not useful: accuracy is 44.80% with a **51.3% tie rate**. At the embedding stage, identical last tokens—often punctuation—have identical vectors and contain no sentence context.
- After layer 0, accuracy rises to 70.20% as the final token begins incorporating preceding context.
- It reaches 84.10% after layer 1, 90.65% after layer 2, and 91.95% after layer 3.
- The best final-token result is after decoder layer 4: **95.20% accuracy**, 95% interval **94.26–96.14%**, and effect size **1.118**.
- From layer 4 onward, accuracy remains consistently high at approximately 93–95% rather than collapsing in the middle layers.

For the best final-token candidate, split-half cosine is **0.9982**, domain-direction cosine is **0.9529**, and cross-domain accuracies are **94.4%** and **96.1%**. Final-token representations therefore contain a robust and transferable distinction once contextual processing has begun. Because the model is causal, the final real token can attend to the complete preceding sentence. Nevertheless, later controls should evaluate punctuation-matched subsets.

### Shuffled controls

The strongest real directions outperform their shuffled-label controls by large margins:

- Mean pooling after layer 21: 96.65% real versus 60.27% shuffled, a **36.38-point advantage**.
- Mean pooling after layer 1: 95.95% real versus 46.08% shuffled, a **49.87-point advantage**.
- Final-token pooling after layer 4: 95.20% real versus 47.60% shuffled, a **47.60-point advantage**.

The shuffled values vary more than ideal—particularly the 60.27% result—because only three controls were averaged. This does not erase the real result, but a stronger robustness run should use at least 50 balanced sign permutations and report the null distribution and empirical p-value.

### What the experiment establishes

The answer is now stronger than in the six-pair pilot: **Qwen2.5-0.5B contains highly stable linear directions that distinguish held-out formal GYAFC rewrites from their informal sources.** The result generalizes across examples, extraction subsets, and both domains. It is especially strong for mean pooling after decoder layer 21, mean pooling after decoder layers 0–1 and 22, and final-token pooling from approximately decoder layer 4 onward.

The moderate per-pair cosine values show that formalization is not one perfectly uniform operation. The aggregate direction captures a shared component across spelling correction, contraction expansion, slang removal, punctuation, syntax, lexical choice, and possibly semantic or length changes.

It is safest to call this a **GYAFC formalization direction**, not yet a universal or pure formality direction. The experiment demonstrates correlational representation geometry. It does not yet show that adding the vector will causally produce formal text or preserve meaning.

### Recommended steering candidates and controls

For the first causal intervention, compare three layers:

- **Primary:** mean-pooled direction after decoder layer 21—the strongest overall held-out separation.
- **Alternative:** final-token direction after decoder layer 4—strong and stable under a different pooling assumption.
- **Negative layer control:** mean-pooled direction after decoder layer 11—only 52.85% held-out accuracy and effect size 0.084.

Before treating the result as an abstract formality representation, repeat evaluation with more shuffled controls and subsets matched on sentence length, terminal punctuation, spelling corrections, and contractions. Then perform a small positive/negative alpha sweep and measure formality, semantic preservation, fluency, repetition, and degeneration.